In [ ]:
import anndata
import numpy as np

labeled_embeddings = anndata.read_h5ad(snakemake.input.llava_labels)
labeled_embeddings.obs.set_index("orig_ids", inplace=True)  # needed to allow transfer

In [ ]:
adata = anndata.read_h5ad(snakemake.input.full_data)

In [ ]:
adata.obsm["X_single-cellm_umap"] = labeled_embeddings.obsm["X_umap"]
adata.obs["leiden"] = labeled_embeddings.obs["leiden"]
adata.obs["cluster_label"] = labeled_embeddings.obs["cluster_label"]
adata.obs.drop(columns=adata.obs.columns[adata.obs.dtypes == np.int64], inplace=True)
adata.uns["dataset"] = snakemake.wildcards.dataset
adata.uns["model"] = snakemake.wildcards.model


In [ ]:
if 'normalized' in adata.layers:
    adata.X = adata.layers['normalized']
else:
    adata.X = adata.X.log1p()
adata.layers = {}

In [ ]:
adata.write_h5ad(snakemake.output.adata)